# Evaluate Models — DEEP tree (NEW & OLD)

Same analysis as `Evaluate_Models.ipynb`, but on the **deep** models (`model_new_deep` / `model_old_deep`).

Builds one eval frame on the **test** samples:
1. `app` -> `ZEST_KEY`, `appDate`, `flg_thin_file`
2. `target` -> `ZEST_KEY`, `final_DQ60_m24`
3. join app+target on `ZEST_KEY` (inner)
4. **left** join trade -> `trade_months_since_oldest_account_opened__all_accounts`, `trade_count__all_accounts`
5. percent feature pulled from BOTH new & old (suffixed) to flag changed rows
6. merge in deep model scores from `model_new_deep/test_scores.parquet` and `model_old_deep/test_scores.parquet`

Run after `Build_Model_New_Deep` / `Build_Model_Old_Deep` have produced `test_scores.parquet`.

In [1]:
import os, sys
import pandas as pd
sys.path.insert(0, os.getcwd())
import configs
from configs import DATA_DIR

TARGET     = 'final_DQ60_m24'
TRADE_COLS = ['trade_months_since_oldest_account_opened__all_accounts',
              'trade_count__all_accounts']
PCT_COL    = 'trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts'
# evaluate on the TEST samples (the rows test_scores covers)
TEST_DIRS  = [os.path.join(DATA_DIR, 'samples', f'{b}_test')
              for b in ['equifax', 'experian', 'transunion']]
MODELS_DIR = os.path.join(DATA_DIR, 'models')
SUFFIX     = '_deep'   # read model_new_deep / model_old_deep
print('test sample dirs:'); [print('  ', d) for d in TEST_DIRS]
print('models:', [f'model_{v}{SUFFIX}' for v in ('new', 'old')])

test sample dirs:
   /home/jag/payment-processor-research/payment_processing_research_data/samples/equifax_test
   /home/jag/payment-processor-research/payment_processing_research_data/samples/experian_test
   /home/jag/payment-processor-research/payment_processing_research_data/samples/transunion_test
models: ['model_new_deep', 'model_old_deep']


In [3]:
# app: ZEST_KEY + appDate
app = pd.concat([pd.read_parquet(os.path.join(d, 'app.parquet'), columns=['ZEST_KEY', 'appDate'])
                 for d in TEST_DIRS], ignore_index=True)

# target: ZEST_KEY + the actual target
tgt = pd.concat([pd.read_parquet(os.path.join(d, 'target.parquet'), columns=['ZEST_KEY', TARGET])
                 for d in TEST_DIRS], ignore_index=True)

base = app.merge(tgt, on='ZEST_KEY', how='inner')

# trade columns (ZEST_KEY is the index in processed_*, reset to a column on read)
def load_trade_cols(variant, cols):
    parts = []
    for d in TEST_DIRS:
        df = pd.read_parquet(os.path.join(d, f'processed_{variant}'), columns=cols)
        parts.append(df.reset_index())   # ZEST_KEY index -> column
    return pd.concat(parts, ignore_index=True)

# 1) the two NON-percent trade cols (identical in new/old) -- read from new, LEFT join
trade = load_trade_cols('new', TRADE_COLS)
base = base.merge(trade, on='ZEST_KEY', how='left')

# 2) the percent feature, which DIFFERS new vs old -- pull from both, suffixed, LEFT join
PCT_COL = 'trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts'
pct_new = load_trade_cols('new', [PCT_COL]).rename(columns={PCT_COL: PCT_COL + '__new'})
pct_old = load_trade_cols('old', [PCT_COL]).rename(columns={PCT_COL: PCT_COL + '__old'})
base = (base.merge(pct_new, on='ZEST_KEY', how='left')
             .merge(pct_old, on='ZEST_KEY', how='left'))

print('base:', base.shape)
print('cols:', list(base.columns))
base.head()

base: (1200000, 7)
cols: ['ZEST_KEY', 'appDate', 'final_DQ60_m24', 'trade_months_since_oldest_account_opened__all_accounts', 'trade_count__all_accounts', 'trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts__new', 'trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts__old']


,ZEST_KEY,appDate,final_DQ60_m24,trade_months_since_oldest_account_opened__all_accounts,trade_count__all_accounts,trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts__new,trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts__old
0,00004365121_5,2020-02-28,0.0,135.953510,21.0,0.00000,0.00000
1,00017281785_3,2020-03-15,0.0,NaN,NaN,NaN,NaN
2,00024989447_7,2020-01-23,0.0,130.203906,11.0,0.00000,0.00000
3,00001298985_6,2020-03-07,0.0,292.769872,21.0,0.01087,0.01087
4,00014702927_6,2020-02-03,0.0,452.017495,28.0,0.00000,0.00000


In [4]:
base['flg_thin_file'] = (base['trade_months_since_oldest_account_opened__all_accounts'] <= 6) | (base['trade_count__all_accounts'] <= 2)

In [5]:
import numpy as np

mask_missing = (base[PCT_COL + '__new'].isna() | base[PCT_COL + '__old'].isna())
base['file_changed'] = (base[PCT_COL + '__new'] != base[PCT_COL + '__old'])
base.loc[mask_missing, 'file_changed'] = np.nan
print(base['file_changed'].value_counts())
print(base['file_changed'].value_counts(normalize=True))

file_changed
False    1108475
True       75373
Name: count, dtype: int64
file_changed
False    0.936332
True     0.063668
Name: proportion, dtype: float64


In [6]:
# merge in NEW and OLD deep test scores, recording the ACTUAL prediction column per variant
SCORE_COLS = {}   # variant -> the prediction column name in eval_df

def load_scores(variant):
    p = os.path.join(MODELS_DIR, f'model_{variant}{SUFFIX}', 'test_scores.parquet')
    s = pd.read_parquet(p)
    if 'ZEST_KEY' not in s.columns:        # scores are usually indexed by ZEST_KEY
        s = s.reset_index()
    assert 'ZEST_KEY' in s.columns, f'no ZEST_KEY in {p} (cols={list(s.columns)})'
    raw_cols = [c for c in s.columns if c != 'ZEST_KEY']
    s = s.rename(columns={c: f'{c}_{variant}' for c in raw_cols})
    suffixed = [f'{c}_{variant}' for c in raw_cols]
    pred = next((c for c in suffixed if any(k in c.lower() for k in ('score', 'pred', 'prob'))),
                suffixed[0])
    SCORE_COLS[variant] = pred
    print(f'{variant}: {len(s):,} rows | score cols {suffixed} | prediction -> {pred}')
    return s

eval_df = (base
           .merge(load_scores('new'), on='ZEST_KEY', how='left')
           .merge(load_scores('old'), on='ZEST_KEY', how='left'))
print('\neval_df:', eval_df.shape, '| SCORE_COLS =', SCORE_COLS)
eval_df.head()

new: 1,200,000 rows | score cols ['final_model_predictions_new'] | prediction -> final_model_predictions_new
old: 1,200,000 rows | score cols ['final_model_predictions_old'] | prediction -> final_model_predictions_old

eval_df: (1200000, 11) | SCORE_COLS = {'new': 'final_model_predictions_new', 'old': 'final_model_predictions_old'}


,ZEST_KEY,appDate,final_DQ60_m24,trade_months_since_oldest_account_opened__all_accounts,trade_count__all_accounts,trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts__new,trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts__old,flg_thin_file,file_changed,final_model_predictions_new,final_model_predictions_old
0,00004365121_5,2020-02-28,0.0,135.953510,21.0,0.00000,0.00000,False,False,0.059561,0.059786
1,00017281785_3,2020-03-15,0.0,NaN,NaN,NaN,NaN,False,NaN,0.362199,0.362116
2,00024989447_7,2020-01-23,0.0,130.203906,11.0,0.00000,0.00000,False,False,0.066163,0.066717
3,00001298985_6,2020-03-07,0.0,292.769872,21.0,0.01087,0.01087,False,False,0.124862,0.128557
4,00014702927_6,2020-02-03,0.0,452.017495,28.0,0.00000,0.00000,False,False,0.019688,0.020803


In [7]:
# AUC comparison using the ACTUAL prediction column from test_scores (SCORE_COLS)
from sklearn.metrics import roc_auc_score

y = eval_df[TARGET]
print('OVERALL:')
for variant in ['new', 'old']:
    col = SCORE_COLS[variant]
    m = y.notna() & eval_df[col].notna()
    print(f'  {variant}: AUC = {roc_auc_score(y[m], eval_df.loc[m, col]):.4f}  (col={col}, n={m.sum():,})')

print('\nTHIN FILE (flg_thin_file==True):')
tf = eval_df[eval_df['flg_thin_file'] == True]
for variant in ['new', 'old']:
    col = SCORE_COLS[variant]; m = tf[TARGET].notna() & tf[col].notna()
    print(f'  {variant}: AUC = {roc_auc_score(tf.loc[m, TARGET], tf.loc[m, col]):.4f}  (n={m.sum():,})')

print('\nFILE CHANGED (percent feature differs new vs old):')
fc = eval_df[eval_df['file_changed'] == True]
for variant in ['new', 'old']:
    col = SCORE_COLS[variant]; m = fc[TARGET].notna() & fc[col].notna()
    print(f'  {variant}: AUC = {roc_auc_score(fc.loc[m, TARGET], fc.loc[m, col]):.4f}  (n={m.sum():,})')

OVERALL:
  new: AUC = 0.7789  (col=final_model_predictions_new, n=1,200,000)
  old: AUC = 0.7787  (col=final_model_predictions_old, n=1,200,000)

THIN FILE (flg_thin_file==True):
  new: AUC = 0.6812  (n=66,089)
  old: AUC = 0.6813  (n=66,089)

FILE CHANGED (percent feature differs new vs old):
  new: AUC = 0.6944  (n=75,373)
  old: AUC = 0.6952  (n=75,373)


In [8]:
# how much the post-FE matrices differ between new_deep and old_deep
MODELS = os.path.expanduser('~/payment-processor-research/payment_processing_research_data/models')
new = pd.read_parquet(os.path.join(MODELS, f'model_new{SUFFIX}', 'test_fe_data.parquet'))
old = pd.read_parquet(os.path.join(MODELS, f'model_old{SUFFIX}', 'test_fe_data.parquet'))
print('new', new.shape, '| old', old.shape)

cols = [c for c in new.columns if c in old.columns]
new, old = new[cols], old[cols]
X = new.select_dtypes('number').fillna(-1.0).astype('float32')
num_cols = X.columns
Xo = old[num_cols].fillna(-1.0).astype('float32')

rows = []
for c in num_cols:
    neq = ~np.isclose(X[c].values, Xo[c].values, equal_nan=True)
    n = int(neq.sum())
    if n:
        rows.append({'col': c, 'n_changed': n, 'pct_rows': 100*n/len(X),
                     'mean_abs_diff': float(np.abs(X[c].values - Xo[c].values).mean())})
diff = pd.DataFrame(rows).sort_values('n_changed', ascending=False)
ncells = len(X) * len(num_cols)
print(f'{len(diff)} of {len(num_cols)} numeric columns differ '
      f'| {diff["n_changed"].sum():,} of {ncells:,} cells changed '
      f'({100*diff["n_changed"].sum()/ncells:.3f}%)')
print('\nmost-changed columns:')
print(diff.head(20).to_string(index=False))

new (1200000, 1144) | old (1200000, 1144)
1143 of 1143 numeric columns differ | 6,881,899 of 1,371,600,000 cells changed (0.502%)

most-changed columns:
                                                                                                   col  n_changed  pct_rows  mean_abs_diff
                                       trade_mean_percent_of_DQ30_in_last_24_months__all_open_accounts      75373  6.281083       0.001309
                                    trade_mean_percent_of_DQ30_in_last_24_months__active_open_accounts      74374  6.197833       0.001277
                                        trade_max_percent_of_DQ30_in_last_24_months__all_open_accounts      66795  5.566250       0.004309
                                     trade_max_percent_of_DQ30_in_last_24_months__active_open_accounts      66009  5.500750       0.004194
                                 trade_mean_percent_of_DQ30_in_last_24_months__non_derog_open_accounts      65765  5.480417       0.001103
             

In [1]:
import os, gc, numpy as np, pandas as pd
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import roc_auc_score

MODELS = os.path.expanduser('~/payment-processor-research/payment_processing_research_data/models')
TARGET = 'final_DQ60_m24'
SUFFIX = '_deep'          # <-- loads model_new_deep / model_old_deep
FIT_N  = 200_000

# ---- load FE matrices, align ----
new = pd.read_parquet(os.path.join(MODELS, f'model_new{SUFFIX}', 'test_fe_data.parquet'))
old = pd.read_parquet(os.path.join(MODELS, f'model_old{SUFFIX}', 'test_fe_data.parquet'))
cols = [c for c in new.columns if c in old.columns]
keys = new['ZEST_KEY'].values

Xnew = new[cols].select_dtypes('number').fillna(-1.0).astype('float32')
num_cols = Xnew.columns
Xold = old[cols][num_cols].fillna(-1.0).astype('float32')

In [2]:
row_drift = np.abs(Xnew.values - Xold.values).sum(axis=1)

In [3]:
from sklearn.preprocessing import StandardScaler

In [4]:
from sklearn.cluster import KMeans
def simple_kmeans(Xp, k=5, iters=50, seed=0):
  rng = np.random.RandomState(seed)
  Xp = np.ascontiguousarray(Xp, dtype=np.float64)
  centers = Xp[rng.choice(len(Xp), k, replace=False)].copy()
  lab = np.zeros(len(Xp), dtype=np.int32)
  d = np.empty((len(Xp), k))
  for _ in range(iters):
      for j in range(k):
          d[:, j] = ((Xp - centers[j]) ** 2).sum(1)
      lab = d.argmin(1)
      new_c = np.stack([Xp[lab == j].mean(0) if (lab == j).any() else centers[j]
                        for j in range(k)])
      if np.allclose(new_c, centers):
          break
      centers = new_c
  return lab

In [5]:
Xs  = StandardScaler().fit_transform(Xnew.values)
pcs = PCA(n_components=5, random_state=0).fit_transform(Xs)

In [6]:
cluster = simple_kmeans(pcs, k=5, seed=0)

In [7]:
prof = pd.DataFrame({'cluster': cluster, 'row_drift': row_drift})
prof['ZEST_KEY'] = new['ZEST_KEY'].values        # ZEST_KEY is a column; same order as prof

In [8]:
print('cluster sizes:'); print(prof['cluster'].value_counts().sort_index())
print('\nmean absolute NEW-vs-OLD drift per cluster:')

cluster sizes:
cluster
0    221776
1    184858
2    231089
3    252077
4    310200
Name: count, dtype: int64

mean absolute NEW-vs-OLD drift per cluster:


In [9]:
S = os.path.join(os.path.dirname(MODELS), 'samples')
tgt = pd.concat([pd.read_parquet(os.path.join(S, f'{b}_test', 'target.parquet'),
                               columns=['ZEST_KEY', TARGET])
               for b in ['equifax','experian','transunion']], ignore_index=True)

def load_pred(variant):
  s = pd.read_parquet(os.path.join(MODELS, f'model_{variant}{SUFFIX}', 'test_scores.parquet'))
  if 'ZEST_KEY' not in s.columns: s = s.reset_index()
  pcol = [c for c in s.columns if c != 'ZEST_KEY'][0]
  return s[['ZEST_KEY', pcol]].rename(columns={pcol: f'pred_{variant}'})

prof = (prof.merge(tgt, on='ZEST_KEY', how='left')
          .merge(load_pred('new'), on='ZEST_KEY', how='left')
          .merge(load_pred('old'), on='ZEST_KEY', how='left'))
print('matched targets:', prof[TARGET].notna().sum(), 'of', len(prof))

print('\nper-cluster target rate, drift, and AUC new vs old:')
for c, g in prof.groupby('cluster'):
  m = g[TARGET].notna() & g['pred_new'].notna()
  auc_n = roc_auc_score(g.loc[m,TARGET], g.loc[m,'pred_new']) if m.sum() else float('nan')
  auc_o = roc_auc_score(g.loc[m,TARGET], g.loc[m,'pred_old']) if m.sum() else float('nan')
  print(f'cluster {c}: n={len(g):>7,} | bad_rate={g[TARGET].mean():.4f} '
        f'| mean_drift={g["row_drift"].mean():.4f} | AUC new={auc_n:.4f} old={auc_o:.4f}')

matched targets: 1200000 of 1200000

per-cluster target rate, drift, and AUC new vs old:
cluster 0: n=221,776 | bad_rate=0.0684 | mean_drift=0.6714 | AUC new=0.6968 old=0.6966
cluster 1: n=184,858 | bad_rate=0.1730 | mean_drift=1.0114 | AUC new=0.6458 old=0.6460
cluster 2: n=231,089 | bad_rate=0.0329 | mean_drift=0.2638 | AUC new=0.7833 old=0.7828
cluster 3: n=252,077 | bad_rate=0.0549 | mean_drift=0.4129 | AUC new=0.7551 old=0.7552
cluster 4: n=310,200 | bad_rate=0.0534 | mean_drift=0.3278 | AUC new=0.7892 old=0.7889
